# DeepLabV3+ Transfer Learning Inference Demo

DeepLabV3+ with interchangeable backbone transfer learning inference demo

Net pretrained with COCO val2017, then trained with Unreal simulation Images or the fakes generated from the GAN

See: https://pytorch.org/hub/pytorch_vision_deeplabv3_resnet101/

---

Credits ---- Author: **Nate Haddad** nhaddad2112[at]gmail[dot]com

In [ ]:
import os.path as op
import sys

import torch
import numpy as np
from PIL import Image
import yaml

sys.path.append('..')
from utils import vis_segmentation, display_example_pair, vis_grid_4x3, run_inference

In [ ]:
torch.cuda.is_available()

In [ ]:
np.random.seed(42)

In [ ]:
with open('../config/config.yaml', 'r') as f:
    config = yaml.safe_load(f)

Load a previously trained model and run the image through it

In [ ]:
model = torch.load(op.join('..', config['LOAD_MODEL_PATH']))
model.eval();

##### Select your output directory for your inference in the cell below

In [ ]:
output_dir = '/home/pdhegde/semseg_git_fork/semantic-segmentation/out/'

In [ ]:
#### for unreal segmentation


import glob
from PIL import Image
from utils import label_to_color_image


def generate_inference(config, output_dir):
        model = torch.load(op.join('..', config['LOAD_MODEL_PATH']))
        model.eval()
        data_path = op.join('..', config['DATA_PATH'])
        image_extensions = ('*.jpg', '*.jpeg', '*.png', '*.bmp', '*.gif', '*.tiff')
        image_list = []
        for ext in image_extensions:
                image_list.extend(glob.glob(op.join(data_path, ext)))

        for img_path in image_list:
                current_image = Image.open(img_path)
                predicted_masks = run_inference(model, current_image)
                #print("Predicted masks unique values:", np.unique(predicted_masks))
                
                seg_img = get_seg_image(np.array(predicted_masks))
        
                # Construct output path and save the image
                output_path = op.join(output_dir, op.basename(img_path))
                seg_img = Image.fromarray(seg_img)
                seg_img.save(output_path)
                print(f"Saved output to {output_path}")

def get_seg_image(seg_map):
        label_names = np.asarray([
                'sky', 'obstacle', 'vegetation', 'landscape_terrain',
        ])
        full_label_map = np.arange(len(label_names)).reshape(len(label_names), 1)
        full_color_map = label_to_color_image(full_label_map)
        seg_image = label_to_color_image(seg_map).astype(np.uint8)

        return seg_image


In [ ]:
generate_inference(config, output_dir)